***Programación para la ciencia de datos.***

***Profesor: Pablo Espinoza Quilaqueo***

***Estudiantes: Sebastián Gonzalez Cacho, Ignacio Cerda Aguiar, Jesús Morán Toro***

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import plotly.express as px
from IPython.display import display

path = kagglehub.dataset_download("lashagoch/life-expectancy-who-updated")
df = pd.read_csv(path + "/Life-Expectancy-Data-Updated.csv")

100%|██████████| 104k/104k [00:00<00:00, 37.0MB/s]

Extracting files...


**creamos el duplicado del data frame**
**para poder usarlo limpio y tener el otro para poder ver los cambios**

In [ ]:
df_copy = df.copy()

**Revisamos si en nuestro DF nos encotramos con nulos, duplicados o algún error**

In [ ]:
print(df.isnull().sum())
print(df.duplicated().sum())

Country                        0
Region                         0
Year                           0
Infant_deaths                  0
Under_five_deaths              0
Adult_mortality                0
Alcohol_consumption            0
Hepatitis_B                    0
Measles                        0
BMI                            0
Polio                          0
Diphtheria                     0
Incidents_HIV                  0
GDP_per_capita                 0
Population_mln                 0
Thinness_ten_nineteen_years    0
Thinness_five_nine_years       0
Schooling                      0
Economy_status_Developed       0
Economy_status_Developing      0
Life_expectancy                0
dtype: int64
0


**Debido a que nuestro DF se encuentra limpio, procedemos a ensuciarlo con fines de cumplir con lo requerido en la evaluación**

In [ ]:
df = pd.concat([df, df.sample(15, random_state=42)])

np.random.seed(42)
indices_nulos = np.random.choice(df.index, size=25, replace=False)
df.loc[indices_nulos, 'Schooling'] = pd.NA

indices_ruido = df[df['Region'] == 'South America'].sample(10, random_state=42).index
df.loc[indices_ruido, 'Region'] = 'sur america'

print(f"Filas con basura inyectada: {len(df)}")
print(f"Nulos falsos creados en 'Schooling': {df['Schooling'].isnull().sum()}\n")

Filas con basura inyectada: 2879
Nulos falsos creados en 'Schooling': 25



**Ahora procedemos a limpiar estos datos inyectados**

In [ ]:
df = df.drop_duplicates()

df['Region'] = df['Region'].replace({'sur america': 'South America'})

df['Schooling'] = df['Schooling'].fillna(df['Schooling'].median())

print(f"Nulos restantes en Schooling: {df['Schooling'].isna().sum()}")
print(f"Filas duplicadas restantes: {df.duplicated().sum()}")
print(f"Registros con error 'sur america': {len(df[df['Region'] == 'sur america'])}")

Nulos restantes en Schooling: 0
Filas duplicadas restantes: 0
Registros con error 'sur america': 0


**A continuación, visualizaremos outliers que podamos encontrar en algunas columnas**

In [ ]:
def detectar_outliers_iqr(dataframe, columna):
    Q1 = dataframe[columna].quantile(0.25)
    Q3 = dataframe[columna].quantile(0.75)
    IQR = Q3 - Q1

    limite_superior = Q3 + 1.5 * IQR
    limite_inferior = Q1 - 1.5 * IQR

    outliers = dataframe[(dataframe[columna] < limite_inferior) | (dataframe[columna] > limite_superior)]

    return outliers, limite_superior


outliers_infant, lim_sup_calculado = detectar_outliers_iqr(df, 'Infant_deaths')
outliers_adult_mortality, lim_sup_adult = detectar_outliers_iqr(df, 'Adult_mortality')
outliers_alcohol_consumption, lim_sup_alcohol = detectar_outliers_iqr(df, 'Alcohol_consumption')

print(f"Límite superior calculado por la función: {lim_sup_calculado:.2f}")
print(f"Se detectaron {len(outliers_infant)} registros atípicos estadísticamente en mortalidad infantil\n")

print(f"Límite superior calculado por la función: {lim_sup_adult:.2f}")
print(f"Se detectaron {len(outliers_adult_mortality)} registros atípicos estadísticamente en mortalidad adultos\n")

print(f"Límite superior calculado por la función: {lim_sup_alcohol:.2f}")
print(f"Se detectaron {len(outliers_alcohol_consumption)} registros atípicos estadísticament en Consumo de alcohol\n")

display(outliers_infant[['Country', 'Year', 'Infant_deaths']].sort_values(by='Infant_deaths', ascending=False).head(5))
display(outliers_adult_mortality[['Country', 'Year', 'Adult_mortality']].sort_values(by='Adult_mortality', ascending=False).head(5))
display(outliers_alcohol_consumption[['Country', 'Year', 'Alcohol_consumption']].sort_values(by='Alcohol_consumption', ascending=False).head(5))

Límite superior calculado por la función: 106.22
Se detectaron 29 registros atípicos estadísticamente en mortalidad infantil

Límite superior calculado por la función: 456.61
Se detectaron 112 registros atípicos estadísticamente en mortalidad adultos

Límite superior calculado por la función: 17.64
Se detectaron 2 registros atípicos estadísticament en Consumo de alcohol



,Country,Year,Infant_deaths
1417,Sierra Leone,2000,138.1
1815,Sierra Leone,2001,135.6
1923,Sierra Leone,2002,132.9
2803,Sierra Leone,2003,130.2
261,Liberia,2000,127.9


,Country,Year,Adult_mortality
2515,Zimbabwe,2002,719.3605
848,Zimbabwe,2001,703.6770
681,Zimbabwe,2003,703.0000
171,Zimbabwe,2000,687.9930
60,Zimbabwe,2004,686.6390


,Country,Year,Alcohol_consumption
2481,Estonia,2006,17.87
1688,Estonia,2013,17.75


**Con el DF limpio, revisado y habiendo encontrado los outliers nos encargaremos de la fase de transformación de datos.**

In [ ]:
regiones = df_copy['Region'].unique()

print(regiones)

['Middle East' 'European Union' 'Asia' 'South America'
 'Central America and Caribbean' 'Rest of Europe' 'Africa' 'Oceania'
 'North America']


**En esta transformación de datos, nos encargamos de generar un promedio de expectativa de vida con respecto a la región y la distancia que tiene ese país con su región**

In [ ]:
df_copy['promedio region']= df_copy.groupby('Region')['Life_expectancy'].transform('mean')

df_copy['distancia al promedio'] = df_copy['Life_expectancy']-df_copy['promedio region']

print(f" Imprimimos nuestra primera transformación de datos: {df_copy['distancia al promedio']} {df_copy['promedio region']}")

 Imprimimos nuestra primera transformación de datos: 0       2.524554
1       5.084954
2      -4.054861
3      -5.780729
4       7.724554
          ...   
2859   -7.947304
2860   -2.554861
2861    4.845139
2862   -5.915046
2863    7.874583
Name: distancia al promedio, Length: 2864, dtype: float64 0       73.975446
1       77.715046
2       69.454861
3       72.780729
4       73.975446
          ...    
2859    57.847304
2860    69.454861
2861    69.454861
2862    77.715046
2863    74.525417
Name: promedio region, Length: 2864, dtype: float64


**Ahora categorizaremos la expectativa de vida en 3 rangos que posteriormente pasaremos a columnas binarias con pd.getdummy**

In [ ]:
def categorizar_expectativa(valor):
  if valor <= 60:
    return 'Baja'
  elif valor <= 75:
    return 'Media'
  else:
    return 'Alta'
df_copy['Life_expectancy_category'] = df_copy['Life_expectancy'].apply(categorizar_expectativa)

In [ ]:
df_copy = pd.get_dummies(
    df_copy,
    columns=['Life_expectancy_category'],
    dtype=int
  )
df_copy

,Country,Region,Year,Infant_deaths,Under_five_deaths,Adult_mortality,Alcohol_consumption,Hepatitis_B,Measles,BMI,...,Thinness_five_nine_years,Schooling,Economy_status_Developed,Economy_status_Developing,Life_expectancy,promedio region,distancia al promedio,Life_expectancy_category_Alta,Life_expectancy_category_Baja,Life_expectancy_category_Media
0,Turkiye,Middle East,2015,11.1,13.0,105.8240,1.320,97,65,27.8,...,4.8,7.8,0,1,76.5,73.975446,2.524554,1,0,0
1,Spain,European Union,2015,2.7,3.3,57.9025,10.350,97,94,26.0,...,0.5,9.7,1,0,82.8,77.715046,5.084954,1,0,0
2,India,Asia,2007,51.5,67.9,201.0765,1.570,60,35,21.2,...,28.0,5.0,0,1,65.4,69.454861,-4.054861,0,0,1
3,Guyana,South America,2006,32.8,40.5,222.1965,5.680,93,74,25.3,...,5.5,7.9,0,1,67.0,72.780729,-5.780729,0,0,1
4,Israel,Middle East,2012,3.4,4.3,57.9510,2.890,97,89,27.0,...,1.1,12.8,1,0,81.7,73.975446,7.724554,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2859,Niger,Africa,2000,97.0,224.9,291.8240,0.092,72,64,20.8,...,12.9,1.1,0,1,49.9,57.847304,-7.947304,0,1,0
2860,Mongolia,Asia,2009,23.9,28.6,235.2330,6.560,97,97,25.3,...,2.3,9.1,0,1,66.9,69.454861,-2.554861,0,0,1
2861,Sri Lanka,Asia,2004,17.7,28.9,134.8950,1.560,62,95,21.9,...,15.5,10.3,0,1,74.3,69.454861,4.845139,0,0,1
2862,Lithuania,European Union,2002,7.9,9.9,204.0120,11.000,94,95,26.1,...,3.3,11.1,1,0,71.8,77.715046,-5.915046,0,0,1


**A continuación generaremos dos transformaciones que tengan correlación para más adelante poder emplearlas. Una de estas columnas tiene todos sus datos normalizados a cantidad cada 1000 habitantes**

In [ ]:
for col in ['Adult_mortality', 'Under_five_deaths', 'Incidents_HIV']:
  df_copy[col + '_norm'] = (df_copy[col] - df_copy[col].min()) / (df_copy[col].max() - df_copy[col].min())

df_copy['Health_Risk_Score'] = (df_copy['Adult_mortality_norm'] * 0.4) + (df_copy['Under_five_deaths_norm'] * 0.4) + (df_copy['Incidents_HIV_norm'] * 0.2)
df_copy['Health_Risk_Score'].describe()

,Health_Risk_Score
count,2864.000000
mean,0.166483
std,0.154373
min,0.001465
25%,0.052228
50%,0.110050
75%,0.243892
max,0.738199


**En esta transformación lo primeros que hacemos es aplicar np.log1p para la columna de GDP_per_capita. Esto con el fin de normalizar la diferencia de PIB_per_capita entre países desarrollados que son grandes economías vs países con menor PIB, evitando que solo las grandes potencias tengan buen puntaje**

In [ ]:
df_copy['GDP_log'] = np.log1p(df_copy['GDP_per_capita'])

gdp_min = df_copy['GDP_log'].min()
gdp_max = df_copy['GDP_log'].max()
df_copy['GDP_norm'] = (df_copy['GDP_log'] - gdp_min) / (gdp_max - gdp_min)


schooling_min = df_copy['Schooling'].min()
schooling_max = df_copy['Schooling'].max()
df_copy['Schooling_norm'] = (df_copy['Schooling'] - schooling_min) / (schooling_max - schooling_min)

df_copy['Socio_Economic_Index'] = (df_copy['GDP_norm'] * 0.4) + \
                             (df_copy['Schooling_norm'] * 0.4) + \
                             (df_copy['Economy_status_Developed'] * 0.2)
df_copy['Socio_Economic_Index'].describe()

,Socio_Economic_Index
count,2864.000000
mean,0.447336
std,0.235282
min,0.033846
25%,0.260363
50%,0.425068
75%,0.554835
max,0.961425


In [ ]:
correlacion = df_copy['Socio_Economic_Index'].corr(df_copy['Health_Risk_Score'])
print(f"La correlación es: {correlacion:.2f}")

La correlación es: -0.70


**comparamos el dataframe al principio con el dataframe con los cambios realizados**

In [ ]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2864 entries, 0 to 2863
Data columns (total 34 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Country                         2864 non-null   object 
 1   Region                          2864 non-null   object 
 2   Year                            2864 non-null   int64  
 3   Infant_deaths                   2864 non-null   float64
 4   Under_five_deaths               2864 non-null   float64
 5   Adult_mortality                 2864 non-null   float64
 6   Alcohol_consumption             2864 non-null   float64
 7   Hepatitis_B                     2864 non-null   int64  
 8   Measles                         2864 non-null   int64  
 9   BMI                             2864 non-null   float64
 10  Polio                           2864 non-null   int64  
 11  Diphtheria                      2864 non-null   int64  
 12  Incidents_HIV                   28

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2864 entries, 0 to 2863
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Country                      2864 non-null   object 
 1   Region                       2864 non-null   object 
 2   Year                         2864 non-null   int64  
 3   Infant_deaths                2864 non-null   float64
 4   Under_five_deaths            2864 non-null   float64
 5   Adult_mortality              2864 non-null   float64
 6   Alcohol_consumption          2864 non-null   float64
 7   Hepatitis_B                  2864 non-null   int64  
 8   Measles                      2864 non-null   int64  
 9   BMI                          2864 non-null   float64
 10  Polio                        2864 non-null   int64  
 11  Diphtheria                   2864 non-null   int64  
 12  Incidents_HIV                2864 non-null   float64
 13  GDP_per_capita         

**comparamos la expectativa de vida por region donde se puede distinguir visualmente en orden de mejor a peor**

In [ ]:
# Preparar datos: promedio de expectativa de vida por región
df_region = (
    df_copy.groupby('Region', as_index=False)['Life_expectancy']
    .mean()
    .rename(columns={'Life_expectancy': 'Promedio_Vida'})
    .sort_values('Promedio_Vida', ascending=False)
    .round(2)
)

# Gráfico de barras interactivo con Plotly
fig1 = px.bar(
    df_region,
    x='Region',
    y='Promedio_Vida',
    color='Promedio_Vida',
    color_continuous_scale='RdYlGn',
    text='Promedio_Vida',
    title='📊 Expectativa de Vida Promedio por Región del Mundo',
    labels={
        'Promedio_Vida': 'Expectativa de Vida (años)',
        'Region': 'Región'
    }
)

fig1.update_traces(
    texttemplate='%{text:.1f} años',
    textposition='outside'
)

fig1.update_layout(
    xaxis_tickangle=-30,
    yaxis_range=[40, 90],
    coloraxis_showscale=False,
    plot_bgcolor='white',
    title_font_size=16,
    margin=dict(t=80, b=120)
)

fig1.show()

**Muestra como ha cambaido la expextativa de vida por region a traves de los años, tambien permite identificar si las intervenciones han logrado tener un impacto significativo**

In [ ]:
# Preparar datos: promedio anual por región
df_anual = (
    df_copy.groupby(['Year', 'Region'], as_index=False)['Life_expectancy']
    .mean()
    .rename(columns={'Life_expectancy': 'Promedio_Vida'})
    .round(2)
)

# Gráfico de líneas interactivo con Plotly
fig2 = px.line(
    df_anual,
    x='Year',
    y='Promedio_Vida',
    color='Region',
    markers=True,
    title='📈 Evolución de la Expectativa de Vida por Región (por año)',
    labels={
        'Promedio_Vida': 'Expectativa de Vida (años)',
        'Year': 'Año',
        'Region': 'Región'
    }
)

fig2.update_layout(
    plot_bgcolor='white',
    title_font_size=16,
    legend_title_text='Región',
    hovermode='x unified'
)

fig2.update_xaxes(showgrid=True, gridcolor='lightgray')
fig2.update_yaxes(showgrid=True, gridcolor='lightgray')

fig2.show()

In [ ]:
# Recrear la columna de categoría (fue convertida por get_dummies)
def categorizar_expectativa(valor):
    if valor <= 60:
        return 'Baja'
    elif valor <= 75:
        return 'Media'
    else:
        return 'Alta'

df_copy['Life_expectancy_category'] = df_copy['Life_expectancy'].apply(categorizar_expectativa)

In [ ]:
anio_max = df_copy['Year'].max()
df_scatter = df_copy[df_copy['Year'] == anio_max].copy()

fig3 = px.scatter(
    df_scatter,
    x='Socio_Economic_Index',
    y='Life_expectancy',
    color='Life_expectancy_category',
    hover_name='Country',
    color_discrete_map={
        'Alta':  '#2ecc71',
        'Media': '#f39c12',
        'Baja':  '#e74c3c'
    },
    category_orders={'Life_expectancy_category': ['Baja', 'Media', 'Alta']},
    title='Índice Socioeconómico vs Expectativa de Vida',
    labels={
        'Socio_Economic_Index': 'Índice Socioeconómico',
        'Life_expectancy': 'Expectativa de Vida (años)',
        'Life_expectancy_category': 'Categoría'
    }
)

fig3.show()

**mostramos como a mayor indice socieconomico es mayor la expectativa de vida**

In [ ]:
# expectativa de vida por region
fig1.show()

# promedio de expectativa por Region

fig2.show()

#Relacion entre indice socieconomico y expectativa de vida

fig3.show()